In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
staging_schema=f"staging"
dbutils.widgets.text("batch_id","1","BATCH ID")
bronze_customermgmt=f"{catalog_name}.bronze.customermgmt"
bronze_customer=f"{catalog_name}.bronze.customer"
staging_customer=f"{catalog_name}.{staging_schema}.customer_scd2_versions"
silver_batchdate = f"{catalog_name}.silver.batchdate"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
df_bronze_customermgmt=spark.read.table(bronze_customermgmt)


In [0]:
# df_bronze_customermgmt.select("ActionType").distinct().show()

In [0]:
df_bronze_customermgmt=df_bronze_customermgmt.filter(col("ActionType").isin(["NEW", "UPDCUST", "INACT"]))
df_bronze_customermgmt=df_bronze_customermgmt.withColumn("ActionTS",col("ActionTS").cast( "timestamp"))\
.withColumn("C_ID",col("C_ID").cast("BIGINT"))\
.withColumn("C_TIER",expr("try_cast(C_TIER as TINYINT)"))\
.withColumn("C_DOB",col("C_DOB").cast("date"))\
.withColumn("Status", expr("CASE WHEN ActionType = 'INACT' THEN 'INAC' ELSE 'ACTV' END")
)
df_bronze_customermgmt=df_bronze_customermgmt.select("ActionTS","C_ID","Status","C_TAX_ID","C_GNDR","C_TIER","C_DOB","C_L_NAME","C_F_NAME","C_M_NAME","C_ADLINE1","C_ADLINE2","C_ZIPCODE","C_CITY","C_STATE_PROV","C_CTRY","C_PRIM_EMAIL","C_ALT_EMAIL","C_CTRY_1","C_AREA_1","C_LOCAL_1","C_EXT_1","C_CTRY_2","C_AREA_2","C_LOCAL_2","C_EXT_2","C_CTRY_3","C_AREA_3","C_LOCAL_3","C_EXT_3","C_LCL_TX_ID","C_NAT_TX_ID",
"_batch","_run_id")



In [0]:

if batch_id!="1":
    #bactchdate to iterate the Action TS
    batch_date= spark.read.table(silver_batchdate) \
                          .filter(col("batchid") == batch_id) \
                          .select("batchdate").collect()

    df_bronze_customer=spark.read.table(bronze_customer).filter(col("_batch")==batch_id)

    # df_bronze_customer.createOrReplaceTempView("bronze_customer")

    #deduplicate
    # df_bronze_customer=spark.sql("""
    #         with ranked as(
    #             select *,row_number() over(
    #                 partition by C_ID 
    #                 order by cast(CDC_DSN as bigint) desc
    #             ) as rn
    #             from bronze_customer
    #         )
    #         select * except (rn)
    #         from ranked
    #         where rn=1 
    #  """)  
    
    #adding ActionTs column
    df_bronze_customer=df_bronze_customer.withColumn("ActionTS", to_timestamp(lit(batch_date[0][0])))\
    .withColumn("C_ID",col("C_ID").cast("BIGINT"))\
    .withColumn("C_TIER",expr("try_cast(C_TIER as TINYINT)"))\
    .withColumn("C_DOB",col("C_DOB").cast("date"))\
    .withColumnRenamed("C_EMAIL_1", "C_PRIM_EMAIL")\
    .withColumnRenamed("C_EMAIL_2", "C_ALT_EMAIL")\
    .withColumnRenamed("C_ST_ID", "Status")

    
    df_bronze_customer=df_bronze_customer.select("ActionTS","C_ID","Status","C_TAX_ID","C_GNDR","C_TIER","C_DOB","C_L_NAME","C_F_NAME","C_M_NAME","C_ADLINE1","C_ADLINE2","C_ZIPCODE","C_CITY","C_STATE_PROV","C_CTRY","C_PRIM_EMAIL","C_ALT_EMAIL","C_CTRY_1","C_AREA_1","C_LOCAL_1","C_EXT_1","C_CTRY_2","C_AREA_2","C_LOCAL_2","C_EXT_2","C_CTRY_3","C_AREA_3","C_LOCAL_3","C_EXT_3","C_LCL_TX_ID","C_NAT_TX_ID",
    "_batch","_run_id")

    df_combined=df_bronze_customermgmt.union(df_bronze_customer)
else:
    df_combined=df_bronze_customermgmt

In [0]:
#window function
window=Window.partitionBy("C_ID")\
    .orderBy("ActionTS")\
    .rowsBetween(Window.unboundedPreceding,Window.currentRow)

#this column will user for forward fill
columns_to_fill=[
    "Status","C_TAX_ID", "C_GNDR", "C_TIER", "C_DOB", "C_L_NAME", "C_F_NAME", "C_M_NAME", 
    "C_ADLINE1", "C_ADLINE2", "C_ZIPCODE", "C_CITY", "C_STATE_PROV", "C_CTRY", 
    "C_PRIM_EMAIL", "C_ALT_EMAIL", "C_CTRY_1", "C_AREA_1", "C_LOCAL_1", "C_EXT_1", 
    "C_CTRY_2", "C_AREA_2", "C_LOCAL_2", "C_EXT_2", "C_CTRY_3", "C_AREA_3", 
    "C_LOCAL_3", "C_EXT_3", "C_LCL_TX_ID", "C_NAT_TX_ID"
]

for c in columns_to_fill:
    df_combined=df_combined.withColumn(c,last(col(c),ignorenulls=True).over(window))

In [0]:
scd_window=Window.partitionBy("C_ID")\
    .orderBy("ActionTS")

df_combined=df_combined.withColumn("EffectiveDate",col("ActionTS").cast("date"))\
    .withColumn("EndDate",lead(col("ActionTS").cast("date")).over(scd_window))\
    .withColumn("EndDate",coalesce(col("EndDate"),to_date(lit("9999-12-31"))))\
    .withColumn("IsCurrent",col("EndDate")==lit("9999-12-31")) \
    .withColumn("version_number", row_number().over(scd_window))

In [0]:
df_combined = df_combined.withColumn("record_hash", md5(concat_ws("||", *columns_to_fill)))


In [0]:
print(df_combined.count())

In [0]:
try:
    print(f"starting Write to {staging_customer}")
    df_combined.write.format('delta').mode("overwrite").saveAsTable(staging_customer)
    count=df_combined.count()
    print(f"Write to {staging_customer} completed with {count}")

    run_id=df_combined.select("_run_id").first()[0]
    staging_history = spark.sql(f"DESCRIBE HISTORY {staging_customer}").first()
    target_count = int(staging_history["operationMetrics"].get("numOutputRows", 0))

    source_count = target_count

    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="customer_scd2_versions",
        source_layer="bronze",      
        target_layer="staging",      
        source_count=source_count,
        target_count=target_count
    )
    
    # 4. Log Audit Event
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id,
        layer="staging",
        table_name="customer_scd2_versions",
        operation="OVERWRITE",      
        rows_affected=target_count
    )
except Exception as e:
    print(f"Error in writing to {staging_customer} : {e}")